                               #######             
                              #                    
        ############ -#####+.##     ######         |	DataDrivenConstruction.io
        ##+########## .-##+ #-   .+########        |	
        ####     ####.     ..    #      ###        |	Dive into the future of flexibility in processing data 
        ####     .#+#.     #    .# .               |	without using BIM tools!
        ####     -#+#.    #.     .#.               |	
        ####      #+#.   ##-                       |	
        ####     ####        #.  #     -####       |	
        ############  .###+ #. ############        |
        #########.   +#### #+.   ########          |	If you have any questions, concerns or special requests,
                           #+                      |	please contact us at info@datadrivenconstruction.io	
                          #                        
                         #  

In [ ]:
import os, sys, tempfile
from pathlib import Path

# Folder of this repository (it holds DDC_WINDOWS_Converters, ddc-tools and Sample_Projects)
REPO = Path(os.environ.get("DDC_REPO", r"C:\DDC\cad2data-Revit-IFC-DWG"))
sys.path.insert(0, str(REPO / "ddc-tools" / "src"))   # or install once:  pip install "<repo>\ddc-tools"
from ddc_tools import convert, table

# Input: a model file (.rvt, .ifc, .dwg) or the folder of a converter result.
# The default is the open IFC sample of this repository.
model = os.environ.get("DDC_MODEL", str(REPO / "Sample_Projects" / "Ifc2x3_Duplex_Architecture.ifc"))
OUT = Path(os.environ.get("DDC_OUT", Path(tempfile.gettempdir()) / "ddc_notebooks"))
OUT.mkdir(parents=True, exist_ok=True)

# Model file -> converter -> flat table with the columns of the old *_rvt.xlsx / *_ifc.xlsx
# (the same as the command line:  ddc convert <model> --xlsx <file>)
xlsx = str(OUT / "model_table.xlsx")
if os.path.isfile(model) and model.lower().endswith((".rvt", ".rfa", ".dwg")):
    convert.convert(model, work=str(OUT / (Path(model).stem + ".converted")), xlsx=xlsx)
else:                                    # an .ifc file or a converter output folder
    table.write_xlsx(table.build_table(model), xlsx)
print("table:", xlsx)

# Importing Revit and IFC data
import pandas as pd
df = pd.read_excel(xlsx, sheet_name="Project_Data")

# Revit tables hold OST_ category codes, IFC tables hold IFC classes
WALLS = ['OST_Walls', 'IfcWall', 'IfcWallStandardCase']
WINDOWS = ['OST_Windows', 'IfcWindow']
df

In [ ]:
# Filtering data in Revit and IFC projects

df[df['Category'].isin(WALLS + WINDOWS)]

In [ ]:
# Create a basic pie chart

df.groupby('Type')['Length'].sum().plot.pie()

In [ ]:
#Regular expression in Revit and IFC

df[df['Category'].str.match('OST_Wal|IfcWal')]

In [ ]:
# Grouping a Revit or IFC project by parameters

df.groupby('Category')[['Volume', 'Length']].sum()

In [ ]:
#QTO - Finding volumetric quantities for the group

dfq= df[df['Category'].str.match('OST_Wal|IfcWal')]
dfq = dfq.groupby('Category')[['Volume', 'Length']].sum()
dfq

In [ ]:
# Bar plot can be created as follows

dfp = df.groupby('Category')['Volume'].sum()
dfp.plot(kind='barh')


In [ ]:
# Grouping with a quantity

df_groups = df.groupby(['Category'])['Volume'].agg(['sum', 'count'])
df_groups

In [ ]:
# Creating a grouping and saving as Excel

dfe = df.groupby(['Category'])['Length'].agg(['sum', 'count'])
dfe.to_excel(OUT / "output.xlsx", sheet_name='Category_estimate')

In [ ]:
# Replacing values in parameters

df.Category = df.Category.replace(WALLS, "Wände")
df

In [ ]:
# Data filtering option

f_filter = df['Category']=='Wände'
df[f_filter]

In [ ]:
# Purely integer-location based indexing for selection by position

df.iloc[2:4, 0:3]

In [ ]:
# PDF creation library installation (once):  pip install fpdf2

In [ ]:
from fpdf import FPDF

# Determining the volumetric characteristics of the group
s_cat = 'OST_Windows|IfcWindow'
dfq= df[df['Category'].str.match(s_cat)]
dfq = dfq.groupby('Category')[['Volume', 'Length']].sum()
cat_len = str(round(dfq.iloc[0]['Length'], 3))
cat_vol = str(round(dfq.iloc[0]['Volume'], 3))

# Creating a PDF document based on the parameters found
pdf = FPDF()
pdf.add_page()
pdf.set_font('Helvetica', 'B', 16)
pdf.cell(190, 8, 'Category: ' + dfq.index[0], new_x="LMARGIN", new_y="NEXT")
pdf.set_font('Helvetica', '', 14)
pdf.cell(190, 8, 'Sum of volumes: ' + cat_vol, new_x="LMARGIN", new_y="NEXT")
pdf.cell(190, 8, 'Sum of lengths: ' + cat_len, new_x="LMARGIN", new_y="NEXT")

# Saving a document in PDF format
pdf.output(str(OUT / 'Report_DataDrivenConstruction.pdf'))